In [4]:
# --- Python 3.11 env + Kernel ---
!sudo apt-get install -y python3.11 python3.11-venv -qq
!python3.11 -m venv /content/py311env
!/content/py311env/bin/pip install -q --upgrade pip
!/content/py311env/bin/pip install -q ipykernel
!/content/py311env/bin/python -m ipykernel install --user --name py311 --display-name "Python 3.11 (Colab)"

# --- Projekt-Pakete ---
!/content/py311env/bin/pip install -q numpy scanpy scib pooch gdown matplotlib
!/content/py311env/bin/pip install -q torch
!/content/py311env/bin/pip install -q scgpt

Installed kernelspec py311 in /root/.local/share/jupyter/kernels/py311
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


In [2]:
from pathlib import Path
import warnings

import scanpy as sc
import scib
import numpy as np
import sys
import os
import pooch
import gdown
import torch

sys.path.insert(0, "../")

import scgpt as scg
import matplotlib.pyplot as plt

plt.style.context('default')
warnings.simplefilter("ignore", ResourceWarning)

model_dir = Path("./scGPT_CP")

ModuleNotFoundError: No module named 'scanpy'

In [2]:
# Target directory
model_dir = "./scGPT_human"
os.makedirs(model_dir, exist_ok=True)

# Direct file IDs extracted from your links
file_ids = {
    "args.json": "15TEZmd2cZCrHwgfE424fgQkGUZCXiYrR",
    "best_model.pt": "1x1SfmFdI-zcocmqWAd7ZTC9CTEAVfKZq",
    "vocab.json": "1jfT_T5n8WNbO9QZcLWObLdRG8lYFKH-Q",
}

for filename, file_id in file_ids.items():
    output_path = os.path.join(model_dir, filename)

    if filename == "best_model.pt":
        # Clean up corrupted HTML file if it was previously downloaded
        if os.path.exists(output_path) and os.path.getsize(output_path) < 1024 * 1024:
            print(f"Removing corrupted or incomplete {filename}...")
            os.remove(output_path)

        # Check if the valid weights file is already present
        if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            print(f"Skipping {filename} (already exists).")
            continue

        print(f"Downloading {filename} with gdown...")
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url=url, output=output_path, quiet=False)

    else:
        # Pooch natively manages caching and checks if the file exists
        print(f"Downloading/fetching {filename} with pooch...")
        url = f"https://drive.google.com/uc?export=download&id={file_id}"
        pooch.retrieve(
            url=url,
            known_hash=None,
            path=model_dir,
            fname=filename,
            progressbar=True,
        )

print(f"\nFinished! Files are located in {os.path.abspath(model_dir)}")

Downloading/fetching args.json with pooch...
Skipping best_model.pt (already exists).
Downloading/fetching vocab.json with pooch...

Finished! Files are located in /Users/melinariepl/ramming_lab_code/scGPT_human


In [3]:
"""
Calculate the metrics for integration results
"""
def scib_eval(adata, batch_key, cell_type_key, embed_key):
    results = scib.metrics.metrics(
        adata,
        adata_int=adata,
        batch_key=batch_key,
        label_key=cell_type_key,
        embed=embed_key,
        isolated_labels_asw_=False,
        silhouette_=True,
        hvg_score_=False,
        graph_conn_=True,
        pcr_=True,
        isolated_labels_f1_=False,
        trajectory_=False,
        nmi_=True,  # use the clustering, bias to the best matching
        ari_=True,  # use the clustering, bias to the best matching
        cell_cycle_=False,
        kBET_=False,  # kBET return nan sometimes, need to examine
        ilisi_=False,
        clisi_=False,
    )
    result_dict = results[0].to_dict()
    
    # compute avgBIO metrics
    result_dict["avg_bio"] = np.mean(
        [
            result_dict["NMI_cluster/label"],
            result_dict["ARI_cluster/label"],
            result_dict["ASW_label"],
        ]
    )
    
    # compute avgBATCH metrics
    result_dict["avg_batch"] = np.mean(
        [
            result_dict["graph_conn"],
            result_dict["ASW_label/batch"],
        ]
    )
    
    result_dict = {k: v for k, v in result_dict.items() if not np.isnan(v)}
    
    return result_dict

In [4]:
sample_data_path = './data/adata_integrated.h5ad'
adata = sc.read_h5ad(sample_data_path)

gene_col = "Gene Symbol"
cell_type_key = "celltype"
batch_key = "tech"
N_HVG = 2000

In [5]:
adata.var[gene_col] = adata.var.index.values

In [6]:
org_adata = adata.copy()

In [7]:
# preprocess
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
# highly variable genes
# If raw counts are in a layer (e.g., 'counts'):
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_HVG,
    flavor="seurat_v3",
    layer="counts"  # Points directly to the raw counts
)
adata = adata[:, adata.var['highly_variable']]

/Users/melinariepl/miniforge3/envs/scgpt_py311/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
/Users/melinariepl/miniforge3/envs/scgpt_py311/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:391: RuntimeWarning: invalid value encountered in log1p
  np.log1p(x, out=x)


In [8]:
import os
import torch
import scgpt as scg
import scgpt.tasks.cell_emb as cell_emb_mod

# 1. macOS affinity patch
os.sched_getaffinity = lambda _: {0}

# 2. Patch DataLoader workers
_orig_loader = cell_emb_mod.DataLoader
def _forced_zero_workers(*args, **kwargs):
    kwargs["num_workers"] = 0
    return _orig_loader(*args, **kwargs)
cell_emb_mod.DataLoader = _forced_zero_workers

# 3. Run on CPU (Notice device="cpu" passed directly to embed_data)
embed_adata = scg.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    device="cpu",               # <-- Pass explicitly here
    batch_size=32,              # 32 or 64 is recommended for CPU inference
    use_fast_transformer=False,
)

/Users/melinariepl/miniforge3/envs/scgpt_py311/lib/python3.11/site-packages/scgpt/tasks/cell_emb.py:212: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["id_in_vocab"] = [


scGPT - INFO - match 1926/2000 genes in vocabulary of size 60697.


/Users/melinariepl/miniforge3/envs/scgpt_py311/lib/python3.11/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Embedding cells:   0%|          | 0/1552 [00:00<?, ?it/s]/Users/melinariepl/miniforge3/envs/scgpt_py311/lib/python3.11/site-packages/torch/nn/modules/transformer.py:384: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1711403251597/work/aten/src/ATen/NestedTensorImpl.cpp:179.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)
Embedding cells:   1%|          | 9/1552 [02:57<8:26:32, 19.70s/it]


KeyboardInterrupt: 